# Modelagem da Camada Gold: Dimensão Fornecedor

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para importações locais
sys.path.append(str(Path.cwd().parent.parent))

from pyspark.sql import functions as F
from src.modules.spark_session import get_spark_session, close_spark_session
import src.modules.modeling_dim_utils as modeling
import src.modules.utils as utils

In [ ]:
# Inicializa a SparkSession conectada ao cluster do container
spark = get_spark_session("ModelagemGoldDimFornecedor")

# Leitura das tabelas da camada Silver

In [ ]:
# Define caminhos das origens na Silver
silver_estoque_path = "s3a://silver/estoque"

# Lê os dados da Silver definindo como None caso a origem não exista
try:
    df_estoque = spark.read.parquet(silver_estoque_path)
except Exception as e:
    print(f"Aviso: Tabela Silver de Estoque não encontrada: {e}")
    df_estoque = None

# Colunas de Fornecedor na Origem

In [ ]:
# Mostra a pré-visualização das colunas caso a tabela não seja None
if df_estoque is not None:
    print("=== Colunas em Silver Estoque ===")
    display(df_estoque.select("fornecedor").distinct().limit(5).toPandas())

# Cria a Dimensão Fornecedor (dim_fornecedor)

In [ ]:
# Executa a lógica de modelagem unificada
df_dim_fornecedor = modeling.create_dim_fornecedor(df_estoque)

if df_dim_fornecedor is not None:
    # Adiciona a data de carga
    df_dim_fornecedor = df_dim_fornecedor.withColumn("data_carga", F.to_date(F.lit(utils.get_current_date_str())))
    
    # Exibe informações sobre o DataFrame gerado
    print(f"Quantidade total de fornecedores únicos: {df_dim_fornecedor.count()}")
    df_dim_fornecedor.printSchema()
    display(df_dim_fornecedor.limit(10).toPandas())
else:
    print("Nenhum fornecedor foi processado (tabela Silver de Estoque estava ausente).")

In [ ]:
# Finaliza a sessão do Spark
close_spark_session(spark)